# LwM2M — manajemen perangkat di atas CoAP

Server LwM2M dan lampu jalan simulasi berbagi jaringan UDP dalam memori. Lampunya adalah `Lwm2mClient` dengan Device (3), Location (6), Temperature (3303), dan Light Control (3311); arahkan ke server sungguhan dengan `UseServer("host")`.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.21.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.21.0-preview.1"

## Path dan TLV
Setiap nilai punya path: objek / instance / resource / instance resource. TLV mengemasnya dengan ringkas.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Lwm2m;
using IoTCom.Net.Transports;
using System.Net;

var tlv = Convert.FromHexString("C800144F70656E204D6F62696C6520416C6C69616E63658606410001410105");   // from the LwM2M specification
foreach (var v in Lwm2mContent.DecodeTlv(Lwm2mPath.Parse("/3/0"), tlv)) Console.WriteLine(v);
foreach (var f in Lwm2mContent.DescribeTlv(tlv)) Console.WriteLine($"{f.Name,-18} {f.Value}");

## Mendaftar dan membaca
Lampu mendaftar dengan POST /rd; server membaca objek Device sebagai TLV dan Light Control sebagai SenML JSON.

In [ ]:
var lwNet = new InMemoryDatagramNetwork();
var lwServerAddress = new IPEndPoint(IPAddress.Loopback, 5683);
var lwServer = Lwm2mServer.Create(o => o.UseInMemory(lwNet, lwServerAddress));
await lwServer.StartAsync();
var light = Lwm2mStreetLightSimulator.Create(o => { o.UseInMemory(lwNet); o.UseServer(lwServerAddress); });
await light.StartAsync();
var reg = lwServer.Registrations.Single();
Console.WriteLine(reg);
foreach (var v in await lwServer.ReadAsync(reg.Endpoint, Lwm2mPath.Parse("/3/0"))) Console.WriteLine(v);
foreach (var v in await lwServer.ReadAsync(reg.Endpoint, Lwm2mPath.Parse("/3311/0"), Lwm2mFormat.SenMLJson)) Console.WriteLine(v);

## Observe dengan pmin dan pmax
Write-Attributes mengatur seberapa sering notifikasi boleh datang; Observe mengalirkannya.

In [ ]:
var tempPath = Lwm2mPath.Parse("/3303/0/5700");
await lwServer.WriteAttributesAsync(reg.Endpoint, tempPath, pmin: 0, pmax: 5);
var temps = new List<object>();
var observation = await lwServer.ObserveAsync(reg.Endpoint, tempPath, values => temps.Add(values[0].Value));
light.Light.Set(5850, true);
for (var i = 0; i < 3; i++) { light.Step(60); await Task.Delay(300); }
await observation.DisposeAsync();
Console.WriteLine($"initial {observation.Initial[0].Value}, then {string.Join(", ", temps)}");

## Write dan Execute harus diizinkan
Server ini hanya-baca; server kedua dengan AllowWrites() dapat meredupkan lampu dan me-reboot-nya.

In [ ]:
try { await lwServer.WriteAsync(reg.Endpoint, Lwm2mPath.Parse("/3311/0/5851"), 40L); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
await light.DisposeAsync();
await lwServer.DisposeAsync();
var lwAdmin = Lwm2mServer.Create(o => o.UseInMemory(lwNet, lwServerAddress).AllowWrites());
await lwAdmin.StartAsync();
var light2 = Lwm2mStreetLightSimulator.Create(o => { o.UseInMemory(lwNet); o.UseServer(lwServerAddress); });
await light2.StartAsync();
var ep = lwAdmin.Registrations.Single().Endpoint;
await lwAdmin.WriteAsync(ep, Lwm2mPath.Parse("/3311/0/5851"), 40L);
try { await lwAdmin.WriteAsync(ep, Lwm2mPath.Parse("/3311/0/5851"), 150L); } catch (Lwm2mException e) { Console.WriteLine($"150 %: {e.Status}"); }
await lwAdmin.ExecuteAsync(ep, Lwm2mPath.Parse("/3/0/4"));
Console.WriteLine($"dimmer {light2.Dimmer} %, reboots {light2.Reboots}");
await light2.DisposeAsync();
await lwAdmin.DisposeAsync();

## Lebih lanjut
`iotcom lwm2m demo`, `iotcom lwm2m serve --observe /3303/0/5700`, `iotcom lwm2m client --server <host>`, `iotcom lwm2m decode 3/0 <hex>`, *Lampu jalan lewat LwM2M* di Gallery, dan sampel Lwm2mClient. Lihat `docs/id/protocols/lwm2m.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*